In [10]:
%matplotlib inline
import os
import glob
import numpy as np
import monai.transforms as mt
import torch
from tqdm import tqdm
from time import perf_counter as time


import matplotlib.pyplot as plt

from project.dataset_iterable_optimized_release import ZarrIterableDataset, test_plot

In [11]:
# Example usage
batch_size = 4
up_factor = 2
patch_shape = (32, 32, 32)
patch_shape_hr = (64, 64, 64)

HCP_1200_train_paths = glob.glob("../../Vedrana_master_project/3D_datasets/datasets/HCP_1200/ome/train/*.zarr")
HCP_1200_test_paths = glob.glob("../../Vedrana_master_project/3D_datasets/datasets/HCP_1200/ome/test/*.zarr")

IXI_train_paths = glob.glob("../../Vedrana_master_project/3D_datasets/datasets/IXI/ome/train/*.zarr")
IXI_test_paths = glob.glob("../../Vedrana_master_project/3D_datasets/datasets/IXI/ome/test/*.zarr")

dataset_dict = {
    "HCP_1200": {
        "paths": HCP_1200_train_paths,
        "group_pairs": {
            "4": [{"H": "HR/0", "L": "HR/2"}],  # {"H": "HR/1", "L": "HR/3"}
            "2": [{"H": "HR/0", "L": "HR/1"}],  # {"H": "HR/1", "L": "HR/3"}
            "1": [{"H": "HR/0", "L": "HR/0"}],
        },
        "sampling_weight": 1,
        "store_type": "DirectoryStore"
    },
    "IXI": {
        "paths": IXI_train_paths,
        "group_pairs": {
            "4": [{"H": "HR/0", "L": "HR/2"}],  # {"H": "HR/1", "L": "HR/3"}
            "2": [{"H": "HR/0", "L": "HR/1"}],  # {"H": "HR/1", "L": "HR/3"}
            "1": [{"H": "HR/0", "L": "HR/0"}],
        },
        "sampling_weight": 1,
        "store_type": "DirectoryStore"
    }
}

In [12]:

if __name__ ==  '__main__':

    seed = 8883
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Define patch transforms
    patch_transform = mt.Compose([
        mt.Identityd(keys=['H', 'L'], allow_missing_keys=True),
        mt.EnsureChannelFirstd(keys=['H', 'L'], channel_dim='no_channel'),
        # mt.SignalFillEmptyd(keys=['H', 'L'], replacement=0),  # Remove any NaNs
    ])

    dataset = ZarrIterableDataset(dataset_dict,
                                  patch_shape,
                                  patch_shape_hr,
                                  patch_transform,
                                  up_factor=up_factor,
                                  num_workers=1,
                                  queue_size=128,
                                  store_type='DirectoryStore',
                                  num_samples=1000,
                                  sampling_method='random'  # 'random' or 'in_chunk'
                                  )

    num_workers = 1
    persistent_workers = True if num_workers > 0 else False
    dataloader = torch.utils.data.DataLoader(dataset,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            num_workers=num_workers,
                                            pin_memory=False,
                                            persistent_workers=persistent_workers)

    no_epochs = 10
    plot_counter = 0
    plot_interval = 100
    start_time = time()
    for i in range(no_epochs):
        print(f"Epoch {i + 1}/{no_epochs}")
        for batch in tqdm(dataloader, desc='Reconstructing patches\n', mininterval=2):
            # for batch in dataloader:
            pass
            # sleep(0.1)  # Assuming some processing time
            # print("Loaded batch...")
            # for key in batch.keys():
            #     print(f"Key: {key}, Shape: {batch[key].shape}")
            if plot_counter % plot_interval == 0:
                test_plot(batch)
            plot_counter += 1


    time_elapsed = time() - start_time
    print(f"Time taken {time_elapsed} sec.")
    print(f"Time taken per patch {time_elapsed / no_epochs / batch_size} sec. (average)")

    print(f"Loaded {len(dataset)} items from Zarr dataset.")

Epoch 1/10


Reconstructing patches
Reconstructing patches50 [00:00<?, ?it/s]
:   0%|          | 0/250 [01:00<?, ?it/s]


KeyboardInterrupt: 